In [2]:
import pandas as pd

# ------------------------- STEP 1: LOAD DATA -------------------------

# Load patient demographics
patients = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/patients.csv")

# Load hospital admissions
admissions = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/admissions.csv")

# Load ICU stays
icustays = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/icu/icustays.csv")

# Load diagnoses (ICD codes assigned to patients)
diagnoses = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/diagnoses_icd.csv")

# Load prescriptions (medications administered)
prescriptions = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/prescriptions.csv")

# Load medical procedures (ICD codes for treatments)
procedures = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/procedures_icd.csv")

# Load ICD descriptions (for both ICD-9 and ICD-10)
icd_diagnoses = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/d_icd_diagnoses.csv.gz")
icd_procedures = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/d_icd_procedures.csv.gz")

# Convert ICU stay timestamps to datetime format
icustays['intime'] = pd.to_datetime(icustays['intime'], errors='coerce')
icustays['outtime'] = pd.to_datetime(icustays['outtime'], errors='coerce')

# ------------------------- STEP 2: MAP ICD CODES TO DESCRIPTIONS -------------------------

# Merge diagnoses with ICD descriptions
diagnoses = diagnoses.merge(icd_diagnoses, on=["icd_code", "icd_version"], how="left")
diagnoses['diagnosis_description'] = diagnoses['long_title'].fillna("Unknown diagnosis")

# Merge procedures with ICD descriptions
procedures = procedures.merge(icd_procedures, on=["icd_code", "icd_version"], how="left")
procedures['procedure_description'] = procedures['long_title'].fillna("Unknown procedure")

# Keep only necessary columns
diagnoses = diagnoses[['subject_id', 'hadm_id', 'diagnosis_description', 'icd_code']]
procedures = procedures[['subject_id', 'hadm_id', 'procedure_description', 'icd_code']]

# ------------------------- STEP 3: COMPUTE AGE AT EVENTS & MORTALITY -------------------------

# Keep only relevant patient information
patients = patients[['subject_id', 'gender', 'anchor_age', 'anchor_year', 'dod']]

# Merge patient data into all event tables
admissions = admissions.merge(patients, on='subject_id', how='left')
icustays = icustays.merge(patients, on='subject_id', how='left')
diagnoses = diagnoses.merge(patients, on='subject_id', how='left')
procedures = procedures.merge(patients, on='subject_id', how='left')
prescriptions = prescriptions.merge(patients, on='subject_id', how='left')

# Convert date columns
admissions['admittime'] = pd.to_datetime(admissions['admittime'])
prescriptions['starttime'] = pd.to_datetime(prescriptions['starttime'])
patients['dod'] = pd.to_datetime(patients['dod'], errors='coerce')

# Compute age at event time
admissions['age_at_event'] = admissions['anchor_age'] + (admissions['admittime'].dt.year - admissions['anchor_year'])
prescriptions['age_at_event'] = prescriptions['anchor_age'] + (prescriptions['starttime'].dt.year - prescriptions['anchor_year'])

# Assign age at death for deceased patients
patients['age_at_death'] = patients['anchor_age'] + (patients['dod'].dt.year - patients['anchor_year'])
patients['death_flag'] = patients['dod'].notna().astype(int)

# Merge mortality data into admissions
admissions = admissions.merge(patients[['subject_id', 'death_flag', 'age_at_death']], on='subject_id', how='left')

# ------------------------- STEP 4: BUILD TABULAR DATASET -------------------------

# Select main features
features = admissions[['subject_id', 'hadm_id', 'age_at_event', 'gender', 'admission_type', 'discharge_location', 'death_flag', 'age_at_death']]

# ICU stays: Number of ICU admissions and length of stay per hospitalization
icu_summary = icustays.groupby('hadm_id').agg(
    icu_admissions=('stay_id', 'count'),
    icu_days=('intime', lambda x: (x.max() - x.min()).days)
).reset_index()

# Diagnoses: Count number of diagnoses per hospitalization
diagnosis_summary = diagnoses.groupby('hadm_id').agg(
    num_diagnoses=('icd_code', 'count'),
    diagnosis_list=('diagnosis_description', lambda x: list(x.unique()))
).reset_index()

# Procedures: Count number of procedures per hospitalization
procedure_summary = procedures.groupby('hadm_id').agg(
    num_procedures=('icd_code', 'count'),
    procedure_list=('procedure_description', lambda x: list(x.unique()))
).reset_index()

# Medications: Count number of prescribed drugs per hospitalization
med_summary = prescriptions.groupby('hadm_id').agg(
    num_medications=('drug', 'count'),
    medication_list=('drug', lambda x: list(x.unique()))
).reset_index()

# Merge all features into a single dataset
dataset = features.merge(icu_summary, on='hadm_id', how='left')
dataset = dataset.merge(diagnosis_summary, on='hadm_id', how='left')
dataset = dataset.merge(procedure_summary, on='hadm_id', how='left')
dataset = dataset.merge(med_summary, on='hadm_id', how='left')

# Fill missing values with 0
dataset.fillna({'icu_admissions': 0, 'icu_days': 0, 'num_diagnoses': 0, 'num_procedures': 0, 'num_medications': 0}, inplace=True)
dataset.fillna({'diagnosis_list': 'None', 'procedure_list': 'None', 'medication_list': 'None'}, inplace=True)

# ------------------------- STEP 5: SAVE & DISPLAY DATASET -------------------------

# Save the dataset to a CSV file for ML usage
#dataset.to_csv("mimiciv_clinical_dataset_tabular.csv", index=False)

# Display the dataset
# import ace_tools as tools
# tools.display_dataframe_to_user(name="MIMIC-IV Clinical Dataset (Tabular)", dataframe=dataset)


/tmp/ipykernel_3914/2477701702.py:18: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  prescriptions = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/prescriptions.csv")


In [5]:
admissions[admissions['subject_id']==11530780]

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,...,edregtime,edouttime,hospital_expire_flag,gender,anchor_age,anchor_year,dod,age_at_event,death_flag,age_at_death
82093,11530780,20881796,2150-04-15 22:48:00,2150-04-16 00:20:00,NaN,EU OBSERVATION,P263MT,EMERGENCY ROOM,NaN,Medicaid,...,2150-04-15 15:10:00,2150-04-16 00:20:00,0,F,22,2150,NaN,22,0,NaN
82094,11530780,26728216,2150-04-15 03:25:00,2150-04-16 16:20:00,NaN,DIRECT OBSERVATION,P74T8J,CLINIC REFERRAL,NaN,Medicaid,...,2150-04-15 15:10:00,2150-04-16 00:20:00,0,F,22,2150,NaN,22,0,NaN
82095,11530780,29701295,2150-07-05 23:06:00,2150-07-08 16:20:00,NaN,URGENT,P559SL,PHYSICIAN REFERRAL,HOME,Medicaid,...,NaN,NaN,0,F,22,2150,NaN,22,0,NaN


In [4]:
dataset.head()

,subject_id,hadm_id,age_at_event,gender,admission_type,discharge_location,death_flag,age_at_death,icu_admissions,icu_days,num_diagnoses,diagnosis_list,num_procedures,procedure_list,num_medications,medication_list
0,10000032,22595853,52,F,URGENT,HOME,1,52.0,0.0,0.0,8.0,"[Portal hypertension, Other ascites, Cirrhosis...",1.0,[Percutaneous abdominal drainage],14.0,"[Furosemide, Ipratropium Bromide Neb, Potassiu..."
1,10000032,22841357,52,F,EW EMER.,HOME,1,52.0,0.0,0.0,8.0,[Unspecified viral hepatitis C with hepatic co...,1.0,[Percutaneous abdominal drainage],15.0,"[Furosemide, Rifaximin, Sodium Chloride 0.9% ..."
2,10000032,25742920,52,F,EW EMER.,HOSPICE,1,52.0,0.0,0.0,10.0,[Chronic hepatitis C without mention of hepati...,1.0,[Percutaneous abdominal drainage],28.0,"[Sodium Chloride 0.9% Flush, 0.9% Sodium Chlo..."
3,10000032,29079034,52,F,EW EMER.,HOME,1,52.0,1.0,0.0,13.0,"[Other iatrogenic hypotension, Chronic hepatit...",0.0,None,24.0,"[Bisacodyl, Senna, Calcium Carbonate, Raltegra..."
4,10000068,25022803,19,F,EU OBSERVATION,NaN,0,NaN,0.0,0.0,1.0,"[Alcohol abuse, unspecified]",1.0,[Other nonoperative respiratory measurements],0.0,None


In [2]:
tabular = pd.read_csv("mimiciv_clinical_dataset_tabular.csv")

In [3]:
tabular.head(20)

,subject_id,hadm_id,age_at_event,gender,admission_type,discharge_location,death_flag,age_at_death,icu_admissions,icu_days,num_diagnoses,diagnosis_list,num_procedures,procedure_list,num_medications,medication_list
0,10000032,22595853,52,F,URGENT,HOME,1,52.0,0.0,0.0,8.0,"['Portal hypertension', 'Other ascites', 'Cirr...",1.0,['Percutaneous abdominal drainage'],14.0,"['Furosemide', 'Ipratropium Bromide Neb', 'Pot..."
1,10000032,22841357,52,F,EW EMER.,HOME,1,52.0,0.0,0.0,8.0,['Unspecified viral hepatitis C with hepatic c...,1.0,['Percutaneous abdominal drainage'],15.0,"['Furosemide', 'Rifaximin', 'Sodium Chloride 0..."
2,10000032,25742920,52,F,EW EMER.,HOSPICE,1,52.0,0.0,0.0,10.0,['Chronic hepatitis C without mention of hepat...,1.0,['Percutaneous abdominal drainage'],28.0,"['Sodium Chloride 0.9% Flush', '0.9% Sodium C..."
3,10000032,29079034,52,F,EW EMER.,HOME,1,52.0,1.0,0.0,13.0,"['Other iatrogenic hypotension', 'Chronic hepa...",0.0,None,24.0,"['Bisacodyl', 'Senna', 'Calcium Carbonate', 'R..."
4,10000068,25022803,19,F,EU OBSERVATION,NaN,0,NaN,0.0,0.0,1.0,"['Alcohol abuse, unspecified']",1.0,['Other nonoperative respiratory measurements'],0.0,None
5,10000084,23052089,72,M,EW EMER.,HOME HEALTH CARE,1,73.0,0.0,0.0,6.0,"['Neurocognitive disorder with Lewy bodies', '...",0.0,None,13.0,"['Pramipexole', 'Pravastatin', 'rivastigmine',..."
6,10000084,29888819,72,M,EU OBSERVATION,NaN,1,73.0,0.0,0.0,6.0,"['Altered mental status, unspecified', ""Parkin...",0.0,None,0.0,None
7,10000108,27250926,25,M,EU OBSERVATION,NaN,0,NaN,0.0,0.0,2.0,['Cellulitis and abscess of oral soft tissues'...,0.0,None,0.0,None
8,10000117,22927623,55,F,EU OBSERVATION,NaN,0,NaN,0.0,0.0,9.0,"['Dysphagia, unspecified', 'Other specified sy...",0.0,None,2.0,"['Heparin', 'Sodium Chloride 0.9% Flush']"
9,10000117,27988844,57,F,OBSERVATION ADMIT,HOME HEALTH CARE,0,NaN,0.0,0.0,13.0,['Unspecified intracapsular fracture of left f...,1.0,['Reposition Left Upper Femur with Internal Fi...,27.0,"['Iso-Osmotic Dextrose', 'CeFAZolin', 'Vitamin..."


In [8]:
tabular[tabular['age_at_event'] < 18]

,subject_id,hadm_id,age_at_event,gender,admission_type,discharge_location,death_flag,age_at_death,icu_admissions,icu_days,num_diagnoses,diagnosis_list,num_procedures,procedure_list,num_medications,medication_list


In [10]:
patients['anchor_age'].unique()

array([52, 23, 33, 19, 72, 27, 25, 24, 48, 60, 59, 34, 20, 63, 81, 22, 30,
       53, 32, 74, 29, 86, 18, 54, 80, 65, 73, 40, 70, 64, 46, 55, 28, 43,
       89, 35, 71, 26, 47, 21, 62, 84, 68, 56, 87, 31, 77, 36, 37, 75, 83,
       58, 39, 57, 38, 69, 79, 44, 91, 45, 41, 42, 82, 78, 66, 61, 51, 85,
       76, 88, 49, 50, 67])

In [ ]:
pati